<div align="center">

<h2>ESCUELA POLITÉCNICA NACIONAL</h2>

<h3>FACULTAD DE INGENIERÍA DE SISTEMAS</h3>

<p><strong>Materia:</strong> Recuperación de la Información</p>

<p><strong>Estudiante:</strong> Estéfano Condoy</p>

</div>

<hr>

# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [5]:
# Aquí importo las herramientas para leer el corpus y mostrar las tablas.
import re
from collections import Counter
from math import log, log1p

import nltk
import numpy as np
import pandas as pd
from IPython.display import display
from nltk.corpus import gutenberg

# Aquí cargo los 18 textos de Gutenberg que usé en la práctica anterior.
try:
    ids_documentos = gutenberg.fileids()
except LookupError:
    nltk.download("gutenberg", raise_on_error=True)
    ids_documentos = gutenberg.fileids()

# Aquí reutilizo la tokenización anterior: minúsculas y apóstrofos internos.
TOKEN_RE = re.compile(r"[^\W_]+(?:'[^\W_]+)?", flags=re.UNICODE)

def tokenizar(texto):
    return TOKEN_RE.findall(texto.lower().replace("\u2019", "'"))

# Aquí guardo el texto original para los fragmentos y los tokens para el cálculo.
textos_originales = {}
tokens_documentos = {}
titulos = {}
filas_corpus = []
for id_doc in ids_documentos:
    texto = gutenberg.raw(id_doc)
    textos_originales[id_doc] = texto
    tokens_documentos[id_doc] = tokenizar(texto)
    # Aquí extraigo el título del encabezado y quito el autor cuando aparece.
    encabezado = texto.lstrip().splitlines()[0].strip().strip("[]")
    titulos[id_doc] = encabezado.split(" by ", 1)[0].rstrip(",")
    filas_corpus.append({
        "Identificador de documento": id_doc,
        "Título": titulos[id_doc],
        "Número de términos únicos": len(set(tokens_documentos[id_doc])),
    })

# Aquí resumo el tamaño del corpus y la longitud de cada texto en tokens.
n_documentos = len(ids_documentos)
longitudes = np.array([len(tokens_documentos[id_doc]) for id_doc in ids_documentos])
vocabulario_corpus = set().union(*tokens_documentos.values())
display(pd.DataFrame(filas_corpus))
display(pd.DataFrame([{
    "Documentos": n_documentos,
    "Tamaño del vocabulario": len(vocabulario_corpus),
    "Longitud mínima (tokens)": longitudes.min(),
    "Longitud media (tokens)": round(longitudes.mean(), 2),
    "Longitud máxima (tokens)": longitudes.max(),
}]))

,Identificador de documento,Título,Número de términos únicos
0,austen-emma.txt,Emma,7211
1,austen-persuasion.txt,Persuasion,5861
2,austen-sense.txt,Sense and Sensibility,6431
3,bible-kjv.txt,The King James Bible,12965
4,blake-poems.txt,Poems,1544
5,bryant-stories.txt,Stories to Tell to Children,3915
6,burgess-busterbrown.txt,The Adventures of Buster Bear,1549
7,carroll-alice.txt,Alice's Adventures in Wonderland,2630
8,chesterton-ball.txt,The Ball and The Cross,8341
9,chesterton-brown.txt,The Wisdom of Father Brown,7884


,Documentos,Tamaño del vocabulario,Longitud mínima (tokens),Longitud media (tokens),Longitud máxima (tokens)
0,18,43825,6839,121225.78,852256


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [12]:
# Aquí calculo TF contando todas las apariciones de cada término.
def calcular_tf(tokens):
    return Counter(tokens)

# Aquí sumo uno al DF por cada documento que contiene el término.
def calcular_df(conteos_documentos):
    frecuencias = Counter()
    for conteos in conteos_documentos.values():
        frecuencias.update(conteos.keys())
    return frecuencias

# Aquí uso el logaritmo natural para el IDF sin suavizado.
def calcular_idf(df, total_documentos):
    return log(total_documentos / df) + 1

# Aquí multiplico la frecuencia del término por su IDF.
def calcular_peso(tf, idf):
    return tf * idf

# Aquí obtengo los conteos de todos los documentos antes de hacer los ejemplos.
tf_por_documento = {
    id_doc: calcular_tf(tokens_documentos[id_doc])
    for id_doc in ids_documentos
}
df_corpus = calcular_df(tf_por_documento)

# Aquí muestro seis pesos TF-IDF con tres términos y dos novelas de Austen.
ejemplos = []
for id_doc in ("austen-emma.txt", "austen-sense.txt"):
    for termino in ("marriage", "family", "fortune"):
        tf = tf_por_documento[id_doc][termino]
        df = df_corpus[termino]
        idf = calcular_idf(df, n_documentos)
        ejemplos.append({
            "Documento": id_doc,
            "Término": termino,
            "TF": tf,
            "DF": df,
            "IDF": idf,
            "TF-IDF": calcular_peso(tf, idf),
        })

display(pd.DataFrame(ejemplos))

,Documento,Término,TF,DF,IDF,TF-IDF
0,austen-emma.txt,marriage,35,12,1.405465,49.191279
1,austen-emma.txt,family,77,13,1.325422,102.057525
2,austen-emma.txt,fortune,39,12,1.405465,54.813139
3,austen-sense.txt,marriage,44,12,1.405465,61.840465
4,austen-sense.txt,family,83,13,1.325422,110.010059
5,austen-sense.txt,fortune,38,12,1.405465,53.407674


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [15]:
# Aquí ordeno el vocabulario para que cada término tenga una columna fija.
vocabulario = sorted(vocabulario_corpus)
columnas = {termino: i for i, termino in enumerate(vocabulario)}

# Aquí reutilizo los conteos TF y DF de la celda anterior y calculo el IDF del corpus.
idf_corpus = {
    termino: calcular_idf(df_corpus[termino], n_documentos)
    for termino in vocabulario
}

# Aquí armo la matriz: las filas son documentos y las columnas son términos.
# Empiezo con ceros para que los términos ausentes queden con peso cero.
tfidf_manual = np.zeros((n_documentos, len(vocabulario)))
for fila, id_doc in enumerate(ids_documentos):
    for termino, tf in tf_por_documento[id_doc].items():
        columna = columnas[termino]
        tfidf_manual[fila, columna] = calcular_peso(tf, idf_corpus[termino])

print(f"Matriz TF-IDF manual: {tfidf_manual.shape[0]} documentos × "
      f"{tfidf_manual.shape[1]:,} términos.")

Matriz TF-IDF manual: 18 documentos × 43,825 términos.


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.